In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings from messy raw data
import warnings
warnings.filterwarnings("ignore")

In [ ]:
avonet = pd.read_csv("../data/processed/avonet_FE_01.csv")

In [ ]:
avonet.columns

In [ ]:
morphological_cols = [
    "beak_culmen", "beak_nares", "beak_width", "beak_depth",
    "beak_elongation", "beak_robustness",
    "tarsus",
    "wing_len", "kipps", "secondary", "hwi",
    "wing_pointedness", "wing_loading", "log_wing_loading", "aspect_ratio",
    "tail", "tail_to_wing",
    "mass", "log_mass", "body_condition",
]

dimorphism_col = [  "dimorphism_beak_culmen", "dimorphism_beak_nares", "dimorphism_beak_width",
    "dimorphism_tarsus",
    "dimorphism_wing_len", "dimorphism_kipps", "dimorphism_secondary", "dimorphism_hwi",
    "dimorphism_tail" ]


ecological_cols = [
    "habitat",
    "habitat_density",
    "migration",
    "trophic_level",
    "trophic_niche",
    "lifestyle"
]

geographical_cols = [
    "lat_min", "abs_lat_min",
    "lat_max", "abs_lat_max",
    "lat_centroid", "abs_lat_centroid",
    "lon_centroid",
    "range_size", "log_RangeSize",
    "climate_zone"
]

taxonomy_cols = [
    "species_birdtree",
    "family_birdlife", "order_birdlife",
    "family_birdtree", "order_birdtree"
]

metadata_cols = [
    "avibase_id",
    "mass_source",
    "inference",
    "total_individuals",
    "female_count",
    "male_count"
]

# section-1  : (analyse feature deeply)

In [ ]:
df= avonet.copy()

In [ ]:
grouping_cols = [
    "order_birdlife",
    "trophic_level",
    "migration",
    "habitat_density"
]

In [ ]:
for col in morphological_cols:
    if col not in df.columns:
        continue

    print("\n" + "="*50)
    print(f"Feature: {col}")
    print("="*50)

    # -------------------------
    # 1. Basic Statistics
    # -------------------------
    print(df[col].describe())

    # -------------------------
    # 2. Histogram
    # -------------------------
    fig = px.histogram(df, x=col, title=f"Distribution of {col}")
    fig.update_layout(template="plotly_dark")
    fig.show()

    # -------------------------
    # 3. Grouped Plots
    # -------------------------
    for group in grouping_cols:
        if group not in df.columns:
            continue

        df_temp = df[[col, group]].dropna()

        # limit categories (important for family)
        if group == "family_birdlife":
            top_n = 10
            top_categories = df_temp[group].value_counts().nlargest(top_n).index
            df_temp = df_temp[df_temp[group].isin(top_categories)]

        # -------------------------
        # A. BOX PLOT
        # -------------------------
        fig_box = px.box(
            df_temp,
            x=group,
            y=col,
            title=f"{col} by {group} (Distribution)"
        )

        fig_box.update_traces(
            marker=dict(size=3),
            line=dict(width=1),
            boxpoints="outliers"
        )

        fig_box.update_layout(
            template="plotly_dark",
            xaxis_tickangle=45
        )

        fig_box.show()

        # -------------------------
        # B. BAR PLOT (mean)
        # -------------------------
        df_bar = df_temp.groupby(group)[col].mean().reset_index()

        fig_bar = px.bar(
            df_bar,
            x=group,
            y=col,
            title=f"{col} by {group} (Mean)"
        )

        fig_bar.update_layout(
            template="plotly_dark",
            xaxis_tickangle=45
        )

        fig_bar.show()

In [ ]:


for col in ecological_cols:
    if col not in df.columns:
        continue

    print("\n" + "="*50)
    print(f"Feature: {col}")
    print("="*50)

    # -------------------------
    # 1. Count Plot
    # -------------------------
    df_count = df[col].value_counts().reset_index()
    df_count.columns = [col, "count"]

    fig = px.bar(
        df_count,
        x=col,
        y="count",
        title=f"Count Distribution of {col}"
    )

    fig.update_layout(template="plotly_dark", xaxis_tickangle=45)
    fig.show()

   
    # -------------------------
    # 3. Dominance Insight
    # -------------------------
    most_common = df[col].value_counts().idxmax()
    least_common = df[col].value_counts().idxmin()

    print(f"Most common: {most_common}")
    print(f"Least common: {least_common}")

    # -------------------------
    # 4. Grouped (Stacked)
    # -------------------------
    for group in grouping_cols:
        if group not in df.columns or group == col:
            continue

        df_temp = df[[col, group]].dropna()

        # limit categories for family
        if group == "family_birdlife":
            top_n = 10
            top_categories = df_temp[group].value_counts().nlargest(top_n).index
            df_temp = df_temp[df_temp[group].isin(top_categories)]


        # -------------------------
        # 5. 100% Stacked (better view)
        # -------------------------
        fig = px.histogram(
            df_temp,
            x=group,
            color=col,
            barnorm="percent",
            title=f"{col} % distribution across {group}"
        )

        fig.update_layout(
            template="plotly_dark",
            xaxis_tickangle=45
        )

        fig.show()

In [ ]:
fig = px.scatter_geo(
    df,
    lat="lat_centroid",
    lon="lon_centroid",
    # color="climate_zone",
    hover_name="species_birdtree"
)

fig.update_traces(
    marker=dict(size=1.5, opacity=0.7)
    
)

fig.update_geos(
    projection_scale=1.2,
    showland=True,
    landcolor="rgb(30,30,30)",
)

fig.update_layout(
    template="plotly_dark",
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

# section-2 for dashboard : (relation between features)

In [ ]:
import plotly.express as px

# drop nulls
df_temp = df[["log_mass", "wing_len", "habitat_density"]].dropna()

# scatter plot (numeric vs numeric + color)
fig = px.scatter(
    df_temp,
    x="log_mass",
    y="wing_len",
    # color="habitat_density",
    trendline="ols",
    title="Log Mass vs Wing Length (colored by Habitat Density)"
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Log Mass",
    yaxis_title="Wing Length"
)

fig.show()

In [ ]:
df_num = df[morphological_cols].select_dtypes(include=["number"])

# correlation matrix
corr_matrix = df_num.corr()

corr_pairs = (
    corr_matrix.unstack()
    .reset_index()
)

corr_pairs.columns = ["Feature_1", "Feature_2", "Correlation"]

# remove self-correlation
corr_pairs = corr_pairs[corr_pairs["Feature_1"] != corr_pairs["Feature_2"]]

# remove duplicate pairs
corr_pairs = corr_pairs.drop_duplicates(subset=["Correlation"])

# sort
corr_pairs = corr_pairs.sort_values(by="Correlation", ascending=False)


In [ ]:
fig = px.imshow(
    corr_matrix.round(2),
    text_auto=True,
    aspect="auto",
    title="Correlation Matrix"
)

fig.update_layout(
    template="plotly_dark",
    width=1000,
    height=900,
    font=dict(size=10),
    xaxis_tickangle=60,
    margin=dict(l=50, r=50, t=50, b=100)
)

fig.show()

In [ ]:
df_temp = df[["wing_len", "migration","habitat_density"]].dropna()
fig = px.violin(
    df_temp,
    x="migration",
    y="wing_len",
    color="habitat_density",
    # box=True,           # keep box inside
    points="outliers"   # cleaner
)

fig.update_layout(
    template="plotly_dark",
    title="Wing Length Distribution across Migration (colored by Habitat Density)",
    xaxis_tickangle=45
)

fig.show()

In [ ]:
df_temp = df[["lat_centroid", "lon_centroid", "trophic_niche", "range_size"]].dropna()

fig = px.scatter_geo(
    df,
    lat="lat_centroid",
    lon="lon_centroid",
    color="trophic_niche",
    hover_name="species_birdtree"
)

fig.update_traces(
    marker=dict(size=2, opacity=0.9)
    
)

fig.update_geos(
    projection_scale=1.2,
    showland=True,
    landcolor="rgb(30,30,30)",
)

fig.update_layout(
    template="plotly_dark",
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
from scipy.stats import chi2_contingency

In [ ]:
ct = pd.crosstab(df["migration"], df["habitat"])

In [ ]:
# ── 2. Heatmap ───────────────────────────────────────────────
fig_heatmap = px.imshow(
    ct,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Crosstab Heatmap: Migration vs Species Class",
    labels={"x": "Species Class", "y": "Migration Type", "color": "Count"}
)
fig_heatmap.update_layout(title_x=0.5)
fig_heatmap.show()

In [ ]:
# ── 3. 100% Stacked Bar ──────────────────────────────────────
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct_reset = ct_pct.reset_index().melt(
    id_vars="migration",
    var_name="species_class",
    value_name="percentage"
)

fig_bar = px.bar(
    ct_pct_reset,
    x="migration",
    y="percentage",
    color="species_class",
    title="100% Stacked Bar: Migration vs Species Class",
    labels={"migration": "Migration Type", "percentage": "Percentage (%)"},
    text=ct_pct_reset["percentage"].round(1).astype(str) + "%",
    barmode="stack",
)
fig_bar.update_traces(textposition="inside")
fig_bar.update_layout(title_x=0.5, yaxis_ticksuffix="%")
fig_bar.show()

In [ ]:
# ── 4. Stats ─────────────────────────────────────────────────
chi2, p, dof, _ = chi2_contingency(ct)
n = ct.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

print(f"Chi-Square p-value : {p:.4f} → {'Significant ✅' if p < 0.05 else 'Not Significant ❌'}")
print(f"Cramér's V         : {cramers_v:.3f} → {'Strong' if cramers_v > 0.3 else 'Moderate' if cramers_v > 0.1 else 'Weak'} association")

# ── 5. Dominant Combination ───────────────────────────────────
dominant = ct.stack().idxmax()
print(f"Dominant Pair      : {dominant[0]} + {dominant[1]} (count: {ct.stack().max()})")

# section 3 - relation between 2 species

In [ ]:
df.head()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

radar_cols = ['hwi', 'log_mass', 'aspect_ratio', 'beak_depth', 'body_condition', 'tail_to_wing']
radar_labels = ['HWI', 'Mass', 'Aspect Ratio', 'Beak Depth', 'Body Condition', 'Tail-to-Wing']

COLORS = [
    'rgba(55,138,221,{})',    # blue
    'rgba(216,90,48,{})',     # coral
    'rgba(29,158,117,{})',    # teal
    'rgba(212,83,126,{})',    # pink
    'rgba(239,159,39,{})',    # amber
    'rgba(127,119,221,{})',   # purple
    'rgba(99,153,34,{})',     # green
    'rgba(226,75,74,{})',     # red
]

def plot_species_radar(df, *species_list):
    """
    Usage:
        plot_species_radar(df, "Eagle", "Sparrow")
        plot_species_radar(df, "Eagle", "Sparrow", "Owl", "Falcon")
    """

    # ── Normalize across full dataset ─────────────────────────
    scaler = MinMaxScaler(feature_range=(0, 10))
    scaled = pd.DataFrame(scaler.fit_transform(df[radar_cols]), columns=radar_cols)
    scaled['species_birdtree'] = df['species_birdtree'].values

    labels_closed = radar_labels + [radar_labels[0]]

    fig = go.Figure()

    for i, species in enumerate(species_list):
        color = COLORS[i % len(COLORS)]
        vals = scaled[scaled['species_birdtree'] == species][radar_cols].mean().tolist()

        if not vals:
            print(f"Warning: '{species}' not found in dataset — skipped.")
            continue

        vals_closed = vals + [vals[0]]

        fig.add_trace(go.Scatterpolar(
            r=vals_closed,
            theta=labels_closed,
            fill='toself',
            name=species,
            line=dict(color=color.format(1.0), width=2.5),
            fillcolor=color.format(0.12),
            hovertemplate=(
                "<b>%{theta}</b><br>"
                "Score: %{r:.2f}/10<br>"
                f"Species: {species}<extra></extra>"
            )
        ))

    fig.update_layout(
        # ── Dark theme ────────────────────────────────────────
        template='plotly_dark',
        paper_bgcolor='#0f1117',
        plot_bgcolor='#0f1117',

        # ── Polar config ──────────────────────────────────────
        polar=dict(
            bgcolor='#161b27',
            gridshape='linear',
            radialaxis=dict(
                range=[0, 10],
                tickvals=[2, 4, 6, 8, 10],
                tickfont=dict(size=10, color='rgba(255,255,255,0.35)'),
                gridcolor='rgba(255,255,255,0.08)',
                linecolor='rgba(255,255,255,0.08)',
                tickcolor='rgba(255,255,255,0)',
            ),
            angularaxis=dict(
                tickfont=dict(size=13, color='rgba(255,255,255,0.85)'),
                linecolor='rgba(255,255,255,0.12)',
                gridcolor='rgba(255,255,255,0.08)',
                direction='clockwise',
            ),
        ),

        # ── Title ─────────────────────────────────────────────
        title=dict(
            text=f"Morphological Profile — {' vs '.join(species_list)}",
            x=0.5,
            font=dict(size=16, color='rgba(255,255,255,0.9)'),
        ),

        # ── Legend ────────────────────────────────────────────
        legend=dict(
            orientation='h',
            y=-0.15,
            x=0.5,
            xanchor='center',
            font=dict(size=12, color='rgba(255,255,255,0.75)'),
            bgcolor='rgba(255,255,255,0.05)',
            bordercolor='rgba(255,255,255,0.1)',
            borderwidth=1,
        ),

        margin=dict(t=80, b=80, l=60, r=60),
        height=580,
    )

    fig.show()


# ── Usage ──────────────────────────────────────────────────────
# plot_species_radar(df, "Eagle", "Sparrow")

# plot_species_radar(df, "Eagle", "Sparrow", "Owl")

# plot_species_radar(df, "Eagle", "Sparrow", "Owl", "Falcon", "Hawk")

In [ ]:
plot_species_radar(df, "Abeillia_abeillei", "Abroscopus_albogularis")

In [ ]:
plot_species_radar(df, "Abeillia_abeillei", "Abroscopus_albogularis","Abroscopus_superciliaris")

- their phylogeny distance 
- how much theire measumrent are similar
- show theire dataploitns values in data . 
- and colclusion